# RAG по научным публикациям

Построить поиск по научным публикациям и сравнить ответы локальной языковой модели с ответами на основе найденных фрагментов.

[Описание и данные](README.md) · [История эксперимента](results/recorded_metrics.json)

**Запуск:** Python 3.11 или 3.12. Полное обучение требует данных и подходящего GPU. Выводы очищены; численные результаты ниже появятся после выполнения.

## Подготовка окружения

Ячейка устанавливает зависимости проекта. Локально рекомендуется отдельное виртуальное окружение.

In [ ]:
%pip -q install -r https://raw.githubusercontent.com/Xokage9/ml-portfolio/main/projects/01_rag/requirements.txt


## 1. Сбор корпуса


In [ ]:
import re
import time
from collections import Counter
from pathlib import Path

import arxiv
import faiss
import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rapidfuzz import fuzz, process
from sentence_transformers import SentenceTransformer


In [ ]:
def tidy(text):
    return re.sub(r'\s+', ' ', str(text or '')).strip()


def fetch_articles(target=350):
    client = arxiv.Client(page_size=100, delay_seconds=3.0, num_retries=5)
    rows, seen = [], set()
    queries = [
        'abs:"retrieval-augmented generation" AND (abs:graph OR abs:graphrag)',
        'all:"retrieval augmented generation" AND (all:graph OR all:knowledge)',
    ]
    for query in queries:
        search = arxiv.Search(query=query, max_results=max(target + 120, 470), sort_by=arxiv.SortCriterion.Relevance)
        for attempt in range(3):
            try:
                for paper in client.results(search):
                    doc_id = paper.entry_id.rstrip('/').split('/')[-1].split('v')[0]
                    if doc_id in seen or paper.published.year < 2020:
                        continue
                    row = {
                        'doc_id': doc_id,
                        'title': tidy(paper.title),
                        'authors': '; '.join(a.name for a in paper.authors),
                        'year': int(paper.published.year),
                        'abstract': tidy(paper.summary),
                        'url': paper.entry_id,
                    }
                    if all(row.get(field) for field in ('doc_id', 'title', 'authors', 'abstract', 'url')):
                        rows.append(row)
                        seen.add(doc_id)
                    if len(rows) >= target:
                        break
                break
            except (arxiv.HTTPError, arxiv.UnexpectedEmptyPageError, ConnectionError, TimeoutError) as err:
                if attempt == 2:
                    print('Источник временно недоступен:', str(err)[:180])
                else:
                    time.sleep(3 * (attempt + 1))
        if len(rows) >= target:
            break
    return pd.DataFrame(rows)


Path('rag_lab3_variant1').mkdir(exist_ok=True)
corpus_path = Path('rag_lab3_variant1/articles.jsonl')
if corpus_path.exists():
    articles = pd.read_json(corpus_path, lines=True, dtype={'doc_id': str})
else:
    articles = fetch_articles()
    if len(articles) >= 300:
        articles.to_json(corpus_path, orient='records', lines=True, force_ascii=False)

assert len(articles) >= 300, f'Собрано {len(articles)} статей: нужно минимум 300'
assert articles.doc_id.is_unique
assert articles[['title', 'authors', 'year', 'abstract', 'url']].notna().all().all()
assert (articles.year >= 2020).all()
print(f'Публикаций: {len(articles)} | годы: {articles.year.min()}—{articles.year.max()}')
display(articles[['title', 'year', 'authors', 'url']].head(4))


### 1.1. Проверка корпуса

Пропуски, дубликаты, длина текстов и распределение по годам.


In [ ]:
articles['words'] = (articles.title + ' ' + articles.abstract).str.split().str.len()
quality = pd.DataFrame({
    'показатель': ['публикаций', 'дубликатов ID', 'пустых заголовков', 'пустых аннотаций', 'медиана слов', 'максимум слов'],
    'значение': [len(articles), int(articles.doc_id.duplicated().sum()),
                int(articles.title.eq('').sum()), int(articles.abstract.eq('').sum()),
                round(float(articles.words.median()), 1), int(articles.words.max())]
})
display(quality)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
articles.year.value_counts().sort_index().plot.bar(ax=axes[0], title='Статьи по годам')
articles.words.plot.hist(bins=25, ax=axes[1], title='Длина документа, слов')
plt.tight_layout()
plt.show()


## 2. Чанкинг

Python: 140 слов, перекрытие 25 слов. LangChain: 650 символов, перекрытие 100 символов. Для каждого фрагмента сохраняем источник.


In [ ]:
def chunk_python(text, size=140, overlap=25):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError('Нужно size > 0 и 0 <= overlap < size')
    words = tidy(text).split()
    if not words:
        return []
    if len(words) <= size:
        return [' '.join(words)]
    step, parts, start = size - overlap, [], 0
    while start < len(words):
        end = min(start + size, len(words))
        parts.append(' '.join(words[start:end]))
        if end == len(words):
            break
        start += step
    return parts


splitter = RecursiveCharacterTextSplitter(
    chunk_size=650,
    chunk_overlap=100,
    separators=['\n\n', '. ', '; ', ', ', ' ', ''],
    keep_separator=False,
)


def chunk_langchain(text):
    return splitter.split_text(tidy(text))


def make_chunks(frame, strategy):
    fn = chunk_python if strategy == 'python' else chunk_langchain
    rows = []
    for paper in frame.itertuples(index=False):
        doc_text = f'{paper.title}. {paper.abstract}'
        for number, piece in enumerate(fn(doc_text)):
            rows.append({
                'chunk_id': f'{strategy}:{paper.doc_id}:{number}',
                'doc_id': paper.doc_id,
                'title': paper.title,
                'authors': paper.authors,
                'url': paper.url,
                'text': piece,
            })
    return pd.DataFrame(rows)


chunks = {}
for strategy in ('python', 'langchain'):
    frame = make_chunks(articles, strategy)
    assert len(frame) >= len(articles)
    assert frame.chunk_id.is_unique
    assert frame.doc_id.isin(articles.doc_id).all()
    assert frame[['title', 'authors', 'url', 'text']].notna().all().all()
    chunks[strategy] = frame
    frame.to_json(Path('rag_lab3_variant1') / f'chunks_{strategy}.jsonl', orient='records', lines=True, force_ascii=False)

chunk_stats = pd.DataFrame([
    {'метод': name, 'чанков': len(frame),
     'средняя длина, слов': round(frame.text.str.split().str.len().mean(), 1),
     'макс. длина, слов': int(frame.text.str.split().str.len().max())}
    for name, frame in chunks.items()
])
display(chunk_stats)
display(chunks['python'][['doc_id', 'chunk_id', 'text']].head(2))


## 3. Эмбеддинги и поиск FAISS

Нормализованные векторы MiniLM и отдельный индекс для каждого способа чанкинга. Поиск — по косинусному сходству.


In [ ]:
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda' if torch.cuda.is_available() else 'cpu')
indices = {}
for strategy, frame in chunks.items():
    index_file = Path('rag_lab3_variant1') / f'index_{strategy}.faiss'
    if index_file.exists():
        index = faiss.read_index(str(index_file))
        if index.ntotal != len(frame):
            index = None
    else:
        index = None
    if index is None:
        vectors = embedder.encode(
            frame.text.tolist(), batch_size=32, show_progress_bar=True,
            normalize_embeddings=True, convert_to_numpy=True,
        ).astype('float32')
        index = faiss.IndexFlatIP(vectors.shape[1])
        index.add(vectors)
        faiss.write_index(index, str(index_file))
    assert index.ntotal == len(frame)
    indices[strategy] = index
    print(f'{strategy}: {index.ntotal} векторов, размерность {index.d}')


In [ ]:
def retrieve(query: str, top_k: int = 5, strategy: str = 'python') -> pd.DataFrame:
    if strategy not in indices:
        raise ValueError('Неизвестная стратегия: ' + strategy)
    if not query.strip():
        raise ValueError('Запрос не должен быть пустым')
    top_k = max(1, min(int(top_k), len(chunks[strategy])))
    vector = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, positions = indices[strategy].search(vector, top_k)
    result = chunks[strategy].iloc[positions[0]].copy().reset_index(drop=True)
    result.insert(0, 'rank', range(1, len(result) + 1))
    result['similarity'] = scores[0].astype(float)
    return result


SEARCH_EXAMPLE = 'How are knowledge graphs used in retrieval augmented generation?'
for strategy in ('python', 'langchain'):
    print('Стратегия:', strategy)
    display(retrieve(SEARCH_EXAMPLE, top_k=3, strategy=strategy)[['rank','title','similarity','url']])


## 4. Исправление опечаток

RapidFuzz и словарь корпуса. Аббревиатуры и названия моделей не исправляем.


In [ ]:
PROTECTED = {'rag','llm','faiss','graphrag','gnn','nlp','bm25','bert','gpt','qwen','svd','hnsw','ann','kg'}
WORD_PATTERN = re.compile(r"[A-Za-z]+(?:-[A-Za-z]+)?|[^A-Za-z]+")
COMMON = {'what','which','does','with','from','into','about','compare','how','why','when','where',
          'methods','studies','based','using','and','the','are','can','that','retrieval',
          'augmented','generation','knowledge','graph','graphs','vector','dense','search'}
word_counts = Counter(
    w.lower() for text in (articles.title + ' ' + articles.abstract)
    for w in re.findall(r'[A-Za-z]{4,}', text)
)
vocab = sorted({w for w, freq in word_counts.items() if freq >= 2} | PROTECTED | COMMON)


def edit_distance(a, b):
    from rapidfuzz.distance import Levenshtein
    return Levenshtein.distance(a, b)


def correct_query(query: str) -> str:
    out = []
    for token in WORD_PATTERN.findall(tidy(query)):
        lower = token.lower()
        if not token.isalpha() or len(lower) < 4 or lower in PROTECTED or lower in vocab:
            out.append(token)
            continue
        candidate = process.extractOne(lower, vocab, scorer=fuzz.ratio, score_cutoff=86)
        if candidate is not None and edit_distance(lower, candidate[0]) <= 2:
            out.append(candidate[0])
        else:
            out.append(token)
    return ''.join(out)


TYPO_EXAMPLES = [
    'How does knowlege grph retrival improve RAG?',
    'Compare graph retrievel and dense retrieval',
    'Does FAISS support semantic retrieval for LLM?'
]
for query in TYPO_EXAMPLES:
    print(query, '\n  →', correct_query(query))


## 5. Генерация и проверка ответа

Из найденных аннотаций отбираются относящиеся к вопросу предложения. Phi-4-mini формулирует отдельное утверждение по каждому фрагменту; источник назначает код. Проверяются номера источников и предложений, числа, ограничения и смысловая поддержка моделью DeBERTa. После неудачного повторного ответа выводится точная цитата с явной пометкой.


In [ ]:
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

assert torch.cuda.is_available(), 'Выберите T4 GPU в настройках Colab'
tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-4-mini-instruct')
llm = AutoModelForCausalLM.from_pretrained(
    'microsoft/Phi-4-mini-instruct', device_map='auto', dtype=torch.float16,
    attn_implementation='eager',
)
_ = llm.eval()

nli_tokenizer = AutoTokenizer.from_pretrained('MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli')
nli_model = AutoModelForSequenceClassification.from_pretrained(
    'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli',
    dtype=torch.float16,
).to('cuda').eval()


In [ ]:
import json


def generate_text(messages, max_new_tokens=640):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    encoded = tokenizer(text, return_tensors='pt').to(llm.device)
    with torch.inference_mode():
        output = llm.generate(
            **encoded, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][encoded.input_ids.shape[1]:], skip_special_tokens=True).strip()


def llm_answer(question):
    return generate_text([
        {'role': 'system', 'content': 'You are a scientific assistant. Answer in 2-3 short sentences.'},
        {'role': 'user', 'content': question},
    ], max_new_tokens=220)


def entailment_score(evidence, claim):
    encoded = nli_tokenizer(
        evidence, claim, return_tensors='pt', max_length=512, truncation='only_first',
    ).to(nli_model.device)
    with torch.inference_mode():
        scores = nli_model(**encoded).logits[0].float().softmax(-1)
    return float(scores[0])


def evidence_sentences(text):
    return [part.strip() for part in re.split(r'(?<=[.!?])\s+(?=[A-Z])', tidy(text)) if part.strip()]


def validate_claim(item, sources):
    result = dict(item) if isinstance(item, dict) else {}
    result.update(accepted=False, reason='invalid_schema', entailment=0.0)
    claim, number = result.get('text'), result.get('source')
    if not isinstance(claim, str) or not claim.strip():
        return result
    if re.search(r'\[\d', claim):
        result['reason'] = 'embedded_citation'
        return result
    if type(number) is not int or not 1 <= number <= len(sources):
        result['reason'] = 'invalid_source'
        return result
    paper = sources.iloc[number - 1]
    if 'evidence_ids' in result:
        ids = result['evidence_ids']
        units = evidence_sentences(paper.context_text)
        if not isinstance(ids, list) or not 1 <= len(ids) <= 2 or any(type(i) is not int or not 1 <= i <= len(units) for i in ids):
            result['reason'] = 'invalid_evidence_id'
            return result
        evidence = ' '.join(units[i - 1] for i in dict.fromkeys(ids))
    else:
        evidence = result.get('evidence')
        if not isinstance(evidence, str) or not evidence.strip():
            return result
        normalize = lambda text: re.sub(r'\s+', ' ', text).strip().casefold()
        if normalize(evidence) not in normalize(paper.context_text):
            result['reason'] = 'evidence_not_in_source'
            return result
    result['evidence'] = evidence
    if result.get('mode') == 'extractive' and claim.strip() != evidence.strip():
        result['reason'] = 'modified_quote'
        return result
    if result.get('mode') != 'extractive':
        if re.search(r'\b(?:eliminate[sd]?|prevent[sd]?|guarantee[sd]?)\b', claim, re.I) and not re.search(r'\b(?:aims?|intended|designed|seeks?|goal)\b', claim, re.I):
            result['reason'] = 'unqualified_guarantee'
            return result
        if re.search(r'dataset|benchmark|experiment', evidence, re.I) and re.search(r'outperform|%|performance gain', claim, re.I) and not re.search(r'dataset|benchmark|experiment|evaluat|paper|reported|study', claim, re.I):
            result['reason'] = 'experimental_scope_lost'
            return result
    numbers = lambda text: set(re.findall(r'\d+(?:\.\d+)?', text))
    if not numbers(claim) <= numbers(evidence):
        result['reason'] = 'unsupported_number'
        return result
    for word in ('always', 'every', 'all', 'universally', 'guarantees'):
        if re.search(rf'\b{word}\b', claim, re.I) and not re.search(rf'\b{word}\b', evidence, re.I):
            result['reason'] = 'scope_expansion'
            return result
    result['entailment'] = min(entailment_score(evidence, sentence) for sentence in evidence_sentences(claim))
    result['accepted'] = result['entailment'] >= 0.80
    result['reason'] = 'supported' if result['accepted'] else 'not_entailed'
    return result



def check_and_align_claim(item, sources):
    verdict = validate_claim(item, sources)
    if verdict['accepted'] or verdict['reason'] not in ('not_entailed', 'unsupported_number', 'scope_expansion'):
        return verdict
    units = evidence_sentences(sources.iloc[item['source'] - 1].context_text)
    terms = set(re.findall(r'[a-z0-9]+', item['text'].lower()))
    ranked = sorted(range(len(units)), key=lambda i: len(terms & set(re.findall(r'[a-z0-9]+', units[i].lower()))), reverse=True)
    for i in ranked[:4]:
        for ids in ([i + 1], [i, i + 1] if i else [1]):
            repaired = validate_claim({**item, 'evidence_ids': ids}, sources)
            if repaired['accepted'] and repaired['entailment'] >= 0.95:
                repaired['evidence_repaired'] = True
                repaired['original_evidence_ids'] = item.get('evidence_ids')
                return repaired
    return verdict


def select_evidence(question, sources, limit=3):
    candidates, seen = [], set()
    for number, paper in enumerate(sources.itertuples(index=False), 1):
        if paper.doc_id in seen:
            continue
        seen.add(paper.doc_id)
        units = evidence_sentences(paper.context_text)
        for i, sentence in enumerate(units):
            if 8 <= len(sentence.split()) <= 110:
                candidates.append({'source': number, 'focus_id': i + 1, 'sentence': sentence})
    if not candidates:
        return []
    vectors = embedder.encode(
        [question] + [item['sentence'] for item in candidates],
        normalize_embeddings=True, convert_to_numpy=True,
    )
    scores = vectors[1:] @ vectors[0]
    ranked = sorted(zip(scores, candidates), key=lambda pair: float(pair[0]), reverse=True)
    chosen, used = [], set()
    for score, item in ranked:
        if float(score) < 0.35 or item['source'] in used:
            continue
        units = evidence_sentences(sources.iloc[item['source'] - 1].context_text)
        i = item['focus_id']
        ids = [i - 1, i] if i > 1 else [1]
        chosen.append({**item, 'evidence_ids': ids, 'evidence': ' '.join(units[j - 1] for j in ids), 'relevance': float(score)})
        used.add(item['source'])
        if len(chosen) == limit:
            break
    return chosen


def generate_grounded_claim(question, evidence, feedback=''):
    return generate_text([
        {'role': 'system', 'content': (
            'Answer using only the supplied evidence. Write ONE short factual sentence relevant to the question. '
            'Keep method names and experimental conditions. Begin with The paper describes. Describe what the paper says without claiming that it originated existing methods. Describe study results as results of this paper, '
            'not universal facts. Do not turn a research goal into an achieved result or claim hallucinations are eliminated. Do not add metrics or comparisons absent from the evidence. '
            'No source numbers, no quotation marks, no explanation. If the evidence is irrelevant, write NONE.'
        )},
        {'role': 'user', 'content': f'Question: {question}\nEvidence: {evidence}\n{feedback}'},
    ], max_new_tokens=150)


def rag_answer(question, strategy='python', top_k=5):
    refusal = 'Информация в найденных статьях отсутствует'
    cleaned = correct_query(question)
    found = retrieve(cleaned, top_k=top_k, strategy=strategy)
    found['context_text'] = found.doc_id.map(articles.set_index('doc_id').abstract)
    attempts, history, checks, accepted = [], [], [], []
    if float(found.similarity.max()) < 0.26:
        answer, status = refusal, 'low_similarity'
    else:
        for item in select_evidence(cleaned, found):
            feedback, verdict = '', None
            for attempt in range(2):
                raw = generate_grounded_claim(cleaned, item['evidence'], feedback)
                attempts.append(raw)
                if raw.strip().upper() == 'NONE':
                    break
                verdict = check_and_align_claim({
                    'text': raw, 'source': item['source'], 'evidence_ids': item['evidence_ids'],
                    'mode': 'generated', 'relevance': item['relevance'],
                }, found)
                history.append({'attempt': attempt + 1, 'checks': [verdict]})
                if verdict['accepted']:
                    break
                feedback = f"The previous sentence was not supported ({verdict['reason']}). Use a narrower statement directly stated in the evidence."
            if verdict is None or not verdict['accepted']:
                verdict = validate_claim({
                    'text': item['sentence'], 'source': item['source'],
                    'evidence_ids': [item['focus_id']], 'mode': 'extractive',
                    'relevance': item['relevance'],
                }, found)
                history.append({'fallback': 'exact_source_sentence', 'checks': [verdict]})
            checks.append(verdict)
            if verdict['accepted']:
                accepted.append(verdict)
        if accepted:
            partial = any(item['mode'] == 'extractive' for item in accepted) or len(accepted) != len(checks)
            if re.search(r'\bcompare\b', cleaned, re.I):
                proof = ' '.join(item['evidence'] for item in accepted)
                partial = partial or not (re.search(r'graph', proof, re.I) and re.search(r'vector|dense', proof, re.I))
            status = 'partial_validated' if partial else 'validated'
            answer = '\n'.join(
                (f'«{item["text"].strip()}»' if item['mode'] == 'extractive' else item['text'].strip())
                + f' [{item["source"]}]' for item in accepted
            )
            if partial:
                answer = 'Подтверждённая часть ответа (дословные фрагменты отмечены кавычками):\n' + answer
        else:
            answer, status = 'Не удалось подтвердить ответ по найденным источникам', 'validation_failed'
    return {
        'question': question, 'cleaned_query': cleaned, 'answer': answer,
        'raw_answer': attempts[-1] if attempts else '', 'attempts': attempts,
        'sources': found, 'status': status, 'claims': accepted, 'validation': checks,
        'validation_history': history, 'coverage': 'limited_to_retrieved_abstracts',
        'abstained': status in ('low_similarity', 'validation_failed'),
        'citations_valid': bool(accepted),
    }


In [ ]:
validation_examples = [
    ('Подтверждённый факт', 'TREX combines graph-based and vector-based retrieval techniques.',
     'TREX combines graph and vector retrieval.', None, True),
    ('Подмена метода', 'TREX combines graph-based and vector-based retrieval techniques.',
     'LinearRAG combines graph and vector retrieval.', None, False),
    ('Ложное сравнение', 'Exploration-based methods let an LLM reason about the next hop, but require a separate generation at each hop.',
     'Dense vector retrieval requires a separate generation at each hop.', None, False),
    ('Универсальное превосходство', 'In our experiments, ontology-guided knowledge graphs outperform vector retrieval baselines.',
     'Knowledge graphs always outperform vector retrieval.', None, False),
    ('Неподтверждённая метрика', 'T2RAG improves performance and reduces retrieval costs on six datasets.',
     'T2RAG reduces the hallucination rate.', None, False),
    ('Выдуманный фрагмент', 'TREX combines graph-based and vector-based retrieval techniques.',
     'TREX combines graph and vector retrieval.', 'TREX is guaranteed to be the best.', False),
]
validation_examples.extend([
    ('Цель не равна доказанному результату',
     'Knowledge-graph-driven RAG is an effective strategy to eliminate hallucinations in large language models.',
     'Knowledge graphs eliminate hallucinations in large language models.', None, False),
    ('Потеря условий эксперимента',
     'Evaluated on six datasets, T2RAG outperforms existing methods with performance gains of up to 11%.',
     'T2RAG outperforms existing methods with performance gains of up to 11%.', None, False),
    ('Условия эксперимента сохранены',
     'Evaluated on six datasets, T2RAG outperforms existing methods with performance gains of up to 11%.',
     'The paper reports performance gains of up to 11% for T2RAG on six datasets.', None, True),
])
validation_results = []
for name, premise, claim, quote, expected in validation_examples:
    verdict = validate_claim(
        {'text': claim, 'source': 1, 'evidence': quote or premise},
        pd.DataFrame({'context_text': [premise]}),
    )
    validation_results.append({
        'проверка': name, 'ожидалось': expected, 'принято': verdict['accepted'],
        'причина': verdict['reason'], 'NLI': round(verdict['entailment'], 3),
    })
indexed_sources = pd.DataFrame({'context_text': ['TREX combines graph-based and vector-based retrieval techniques. The paper reports experiments.']})
for name, ids, expected in [
    ('Номер предложения: верный', [1], True),
    ('Номер предложения: отсутствует', [99], False),
    ('Номер предложения: не то доказательство', [2], False),
]:
    verdict = validate_claim({'text': 'TREX combines graph and vector retrieval.', 'source': 1, 'evidence_ids': ids}, indexed_sources)
    validation_results.append({'проверка': name, 'ожидалось': expected, 'принято': verdict['accepted'], 'причина': verdict['reason'], 'NLI': round(verdict['entailment'], 3)})

for name, claim, expected in [
    ('Восстановление верного доказательства', 'TREX combines graph and vector retrieval.', True),
    ('Нельзя восстановить неподтверждённый факт', 'LinearRAG combines graph and vector retrieval.', False),
]:
    verdict = check_and_align_claim({'text': claim, 'source': 1, 'evidence_ids': [2]}, indexed_sources)
    validation_results.append({'проверка': name, 'ожидалось': expected, 'принято': verdict['accepted'], 'причина': verdict['reason'], 'NLI': round(verdict['entailment'], 3)})

validator_tests = pd.DataFrame(validation_results)
display(validator_tests)
assert validator_tests['ожидалось'].eq(validator_tests['принято']).all()


## 6. Сравнение LLM и RAG

Пять запросов: фактический, сравнительный, с опечатками, без ответа и обобщающий.


In [ ]:
TEST_CASES = [
    ('Фактический', 'How are knowledge graphs used in retrieval augmented generation?'),
    ('Сравнение', 'Compare graph-based retrieval with dense vector retrieval in RAG.'),
    ('Опечатки', 'How does knowlege grph retrival improve RAG?'),
    ('Нет ответа', 'What was the exact menu at a Martian school cafeteria on June 12, 1840?'),
    ('Обобщение', 'Summarize the main research trends in graph-based RAG.'),
]

results = []
for category, question in TEST_CASES:
    plain = llm_answer(question)
    for strategy in ('python', 'langchain'):
        out = rag_answer(question, strategy=strategy, top_k=5)
        sources = out['sources']
        results.append({
            'scenario': category,
            'strategy': strategy,
            'question': question,
            'corrected_query': out['cleaned_query'],
            'raw_llm_answer': plain,
            'rag_answer': out['answer'],
            'raw_rag_answer': out['raw_answer'],
            'rag_attempts': out['attempts'],
            'rag_status': out['status'],
            'accepted_claims': out['claims'],
            'claim_validation': out['validation'],
            'validation_history': out['validation_history'],
            'answer_coverage': out['coverage'],
            'citations_valid': out['citations_valid'],
            'generator': 'microsoft/Phi-4-mini-instruct',
            'rag_abstained': out['abstained'],
            'top1_similarity': float(sources.similarity.iloc[0]),
            'source_titles': sources.title.tolist(),
            'source_texts': sources.context_text.tolist(),
            'retrieved_texts': sources.text.tolist(),
            'source_chunk_ids': sources.chunk_id.tolist(),
            'source_urls': sources.url.tolist(),
            'source_scores': sources.similarity.round(4).tolist(),
        })
    print(category, '| готово')

results_df = pd.DataFrame(results)
display(results_df[['scenario', 'strategy', 'corrected_query', 'rag_status', 'top1_similarity']])


In [ ]:
for scenario in results_df.scenario.unique():
    group = results_df[results_df.scenario == scenario]
    print('\n' + '=' * 60 + '\n', scenario, '\n', group.question.iloc[0])
    print('\nБез контекста:\n', group.raw_llm_answer.iloc[0])
    for row in group.itertuples(index=False):
        print(f'\nRAG ({row.strategy}, {row.rag_status}):\n{row.rag_answer}')
        print('Источники:')
        for number, (title, url, score) in enumerate(zip(row.source_titles, row.source_urls, row.source_scores), 1):
            print(f'[{number}] {score:.3f} | {title} | {url}')


## 7. Проверка результатов

Сравниваем ответы, отказы, источники и результат проверки каждого утверждения. Статус validated означает подтверждение утверждений, а не полноту ответа.


In [ ]:
summary = results_df.groupby('strategy', as_index=False).agg(
    scenarios=('scenario', 'count'),
    responses_with_validated_claims=('citations_valid', 'sum'),
    refusals=('rag_abstained', 'sum'),
)
display(summary)
display(pd.crosstab(results_df.strategy, results_df.rag_status))
claim_checks = pd.DataFrame([
    {'scenario': row.scenario, 'strategy': row.strategy, **item}
    for row in results_df.itertuples(index=False) for item in row.claim_validation
], columns=['scenario', 'strategy', 'text', 'source', 'mode', 'accepted', 'reason', 'entailment'])
display(claim_checks[['scenario', 'strategy', 'source', 'mode', 'accepted', 'reason', 'entailment']])
display(pd.crosstab(claim_checks.strategy, claim_checks['mode']))
rejected_drafts = pd.DataFrame([
    {'scenario': row.scenario, 'strategy': row.strategy, 'reason': item['reason']}
    for row in results_df.itertuples(index=False)
    for attempt in row.validation_history for item in attempt['checks'] if not item['accepted']
], columns=['scenario', 'strategy', 'reason'])
if not rejected_drafts.empty:
    display(rejected_drafts.groupby('reason').size().rename('rejected_drafts').to_frame())
assert all(item['accepted'] for row in results_df.accepted_claims for item in row)
assert results_df.loc[results_df.scenario.eq('Нет ответа'), 'rag_answer'].eq('Информация в найденных статьях отсутствует').all()
assert results_df.loc[results_df.scenario.ne('Нет ответа'), 'citations_valid'].all()
for row in results_df.itertuples(index=False):
    for item in row.accepted_claims:
        assert 1 <= item['source'] <= len(row.source_texts)
        assert f"[{item['source']}]" in row.rag_answer
        units = evidence_sentences(row.source_texts[item['source'] - 1])
        assert item['evidence'] == ' '.join(units[i - 1] for i in dict.fromkeys(item['evidence_ids']))
        if item['mode'] == 'extractive':
            assert item['text'] == item['evidence']
print(f'Проверки пройдены: {len(validator_tests)} контрольных примеров; {len(results_df)} сценариев.')
print(f'Утверждений с основаниями: {len(claim_checks)}; отказов вне темы: {int(results_df.rag_abstained.sum())}.')


### 7.1. Вывод

В этой работе я собрал 350 научных статей и сделал систему, которая ищет подходящие материалы и отвечает на вопросы по ним. Для сравнения использовал два способа разбиения текста: собственную функцию на Python и библиотеку LangChain.

В итоговых десяти проверках получилось шесть ответов с подтверждениями из статей, два частичных ответа и два отказа. Оба отказа были на вопросе вне темы собранных материалов. В одном из частичных ответов вместо неудачного пересказа система показала точную цитату и явно её отметила.

Способ разбиения текста повлиял на то, какие статьи попадали в результаты поиска. Например, для сравнения графового и векторного поиска вариант на Python не нашёл прямого сравнения среди пяти выбранных статей, поэтому ответ остался неполным. Вариант с LangChain нашёл статью, где описано сочетание этих подходов. Это показало, что похожая по теме статья не обязательно содержит всё, о чём спрашивает пользователь.

Главный результат работы — ответ можно сверить с конкретными источниками. По сравнению с ответом модели без контекста здесь видно, откуда взялись утверждения. Перед выводом система проверяет их по найденному тексту, а при нехватке сведений ограничивает ответ. Такая проверка помогает отсеять ошибки, хотя и не гарантирует, что любой ответ будет полностью верным и исчерпывающим.


## Как читать результат

350 статей в сохранённом запуске; 657 фрагментов Python и 1024 LangChain. Пять сценариев на каждый вариант. Статус `validated` относится к проверке отдельных утверждений, а не к полноте или безошибочности всего ответа.

Набор из пяти запросов — качественная проверка, а не независимый бенчмарк. Эвристики и NLI-проверка не гарантируют отсутствие галлюцинаций.

Сохранённая история исходного эксперимента находится отдельно в `results/recorded_metrics.json`. Полное GPU-обучение при подготовке этой версии не выполнялось.